In [1]:
# ======================================================================
# PISA 2022 replication dataset builder
#  - loads selected columns from student & school SAS7BDAT files
#  - merges on country & school IDs
#  - filters to six target economies
#  - outputs parquet file for lavaan SEM
# ======================================================================

import pyreadstat
import pandas as pd
import numpy as np

# ----------------------------------------------------------------------
# 1. Define file paths  (EDIT THESE)
# ----------------------------------------------------------------------
student_path = r"C:\Users\mithu\Documents\MEGA\VUW\Summer Research Project\Datasets\PISA 2022\STU_QQQ_SAS\CY08MSP_STU_QQQ.SAS7BDAT"
school_path  = r"C:\Users\mithu\Documents\MEGA\VUW\Summer Research Project\Datasets\PISA 2022\SCH_QQQ_SAS\CY08MSP_SCH_QQQ.SAS7BDAT"

# ----------------------------------------------------------------------
# 2. Define columns to read
# ----------------------------------------------------------------------
student_vars = [
    # IDs
    "CNT", "CNTSCHID",
    # Controls
    "ST001D01T", "ST004D01T", "MISCED", "ESCS",
    # Self-efficacy items
    "ST268Q01JA", "ST268Q04JA", "ST268Q07JA",
    # Plausible values
    "PV1MATH","PV2MATH","PV3MATH","PV4MATH","PV5MATH",
    "PV6MATH","PV7MATH","PV8MATH","PV9MATH","PV10MATH"
]

school_vars = [
    "CNT", "CNTSCHID",
    "SC064Q01TA","SC064Q02TA","SC064Q03TA",
    "SC064Q04NA","SC064Q05WA","SC064Q06WA",
    "SCHSIZE"
]

# ----------------------------------------------------------------------
# 3. Read only those columns (pyreadstat is fast and memory efficient)
# ----------------------------------------------------------------------
stu_df, stu_meta = pyreadstat.read_sas7bdat(student_path, usecols=student_vars)
sch_df, sch_meta = pyreadstat.read_sas7bdat(school_path, usecols=school_vars)

print(f"Loaded student data: {stu_df.shape[0]:,} rows, {stu_df.shape[1]} columns")
print(f"Loaded school data:  {sch_df.shape[0]:,} rows, {sch_df.shape[1]} columns")

Loaded student data: 613,744 rows, 19 columns
Loaded school data:  21,629 rows, 9 columns


In [2]:
stu_df.describe()

,CNTSCHID,ST001D01T,ST004D01T,ST268Q01JA,ST268Q04JA,ST268Q07JA,MISCED,ESCS,PV1MATH,PV2MATH,PV3MATH,PV4MATH,PV5MATH,PV6MATH,PV7MATH,PV8MATH,PV9MATH,PV10MATH
count,6.137440e+05,613744.000000,613665.000000,511028.000000,507679.000000,507614.000000,580302.000000,588276.000000,613744.000000,613744.000000,613744.000000,613744.000000,613744.000000,613744.000000,613744.000000,613744.000000,613744.000000,613744.000000
mean,4.348123e+07,12.331060,1.501749,2.309502,2.341030,3.255836,6.372563,-0.310415,440.874665,440.969491,441.024863,440.946372,440.945821,440.802172,440.879567,440.846367,440.806112,440.898504
std,2.558506e+07,14.990274,0.499997,0.999120,0.935062,0.816343,2.489809,1.129914,101.840726,101.766402,101.845513,101.867824,101.885230,101.784495,101.839511,101.837194,101.821416,101.763870
min,8.000010e+05,7.000000,1.000000,1.000000,1.000000,1.000000,1.000000,-6.840700,0.000000,45.946000,55.543000,0.000000,46.925000,3.281000,52.599000,0.000000,39.804000,33.105000
25%,2.140006e+07,9.000000,1.000000,1.000000,2.000000,3.000000,5.000000,-1.053900,364.325000,364.581750,364.512000,364.451750,364.487750,364.430000,364.500000,364.297000,364.443000,364.569000
50%,3.980055e+07,10.000000,2.000000,2.000000,2.000000,3.000000,7.000000,-0.184500,433.056000,433.170500,433.192500,433.079000,433.167000,432.989000,433.194500,433.084000,432.958000,433.106500
75%,6.880013e+07,10.000000,2.000000,3.000000,3.000000,4.000000,8.000000,0.587400,511.177000,511.253000,511.460000,511.341250,511.207250,511.175250,511.281500,511.139250,510.952000,511.294000
max,9.010018e+07,99.000000,2.000000,4.000000,4.000000,4.000000,10.000000,7.380000,943.041000,933.836000,946.788000,911.464000,896.692000,909.524000,928.840000,912.729000,903.018000,915.231000


In [3]:
sch_df.describe()

,CNTSCHID,SC064Q05WA,SC064Q06WA,SC064Q01TA,SC064Q02TA,SC064Q04NA,SC064Q03TA,SCHSIZE
count,2.162900e+04,19499.000000,19541.000000,19525.000000,19507.000000,18935.000000,19295.000000,18643.000000
mean,4.194972e+07,35.678291,44.209713,38.309398,50.941662,19.877528,25.856232,779.842086
std,2.594669e+07,30.474390,32.193535,30.447242,33.081310,28.469645,31.945060,772.880165
min,8.000010e+05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,2.030020e+07,10.000000,15.000000,10.000000,20.000000,0.000000,3.000000,306.000000
50%,3.980001e+07,25.000000,39.000000,30.000000,50.000000,5.000000,10.000000,600.000000
75%,6.820005e+07,56.000000,73.000000,60.000000,80.000000,28.000000,40.000000,1007.000000
max,9.010018e+07,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,19201.000000


In [4]:
# === 2. Extract column-to-label mappings ===
stu_mapping = pd.DataFrame({
    "Variable_Name": stu_meta.column_names,
    "Variable_Label": stu_meta.column_labels,
    "Dataset_Level": "Student"
})

sch_mapping = pd.DataFrame({
    "Variable_Name": sch_meta.column_names,
    "Variable_Label": sch_meta.column_labels,
    "Dataset_Level": "School"
})

# === 3. Merge mappings ===
combined_mapping = pd.concat([stu_mapping, sch_mapping], ignore_index=True)

# === 4. Save outputs ===
base_dir = r"C:\Users\mithu\Documents\MEGA\VUW\Summer Research Project\Datasets"

stu_mapping.to_csv(f"{base_dir}\\PISA_2022_STUDENT_MAPPING.csv", index=False, encoding="utf-8-sig")
sch_mapping.to_csv(f"{base_dir}\\PISA_2022_SCHOOL_MAPPING.csv", index=False, encoding="utf-8-sig")
combined_mapping.to_csv(f"{base_dir}\\PISA_2022_COMBINED_MAPPING.csv", index=False, encoding="utf-8-sig")

print(f"✅ Mappings saved to: {base_dir}")
print(f"  - Student: PISA_2022_STUDENT_MAPPING.csv ({len(stu_mapping)} variables)")
print(f"  - School:  PISA_2022_SCHOOL_MAPPING.csv ({len(sch_mapping)} variables)")
print(f"  - Combined: PISA_2022_COMBINED_MAPPING.csv ({len(combined_mapping)} total)")

✅ Mappings saved to: C:\Users\mithu\Documents\MEGA\VUW\Summer Research Project\Datasets
  - Student: PISA_2022_STUDENT_MAPPING.csv (19 variables)
  - School:  PISA_2022_SCHOOL_MAPPING.csv (9 variables)
  - Combined: PISA_2022_COMBINED_MAPPING.csv (28 total)


In [5]:
stu_meta.column_labels

['Country code 3-character',
 'Intl. School ID',
 'Student International Grade (Derived)',
 'Student (Standardized) Gender',
 'Agree/disagree: Mathematics is one of my favourite subjects.',
 'Agree/disagree: Mathematics is easy for me.',
 'Agree/disagree: I want to do well in my mathematics class.',
 'Mother’s level of education (ISCED)',
 'Index of economic, social and cultural status',
 'Plausible Value 1 in Mathematics',
 'Plausible Value 2 in Mathematics',
 'Plausible Value 3 in Mathematics',
 'Plausible Value 4 in Mathematics',
 'Plausible Value 5 in Mathematics',
 'Plausible Value 6 in Mathematics',
 'Plausible Value 7 in Mathematics',
 'Plausible Value 8 in Mathematics',
 'Plausible Value 9 in Mathematics',
 'Plausible Value 10 in Mathematics']

In [6]:
sch_meta.column_labels

['Country code 3-character',
 'Intl. School ID',
 "Proportion parent/guardians who: Discussed their child's behaviour with a teacher on the parents' or guardians' own initiative",
 "Proportion parent/guardians who: Discussed their child's behaviour on the initiative of one of their child's teachers",
 "Proportion parent/guardians who: Discussed their child's progress with a teacher on the parents' or guardians' own initiative",
 "Proportion parent/guardians who: Discussed their child's progress on the initiative of one of their child's teachers",
 'Proportion parent/guardians who: Volunteered in physical or extra-curricular activities, (e.g. building maintenance, carpentry, gardening or yard work, school play, sports, field trip)',
 'Proportion parent/guardians who: Participated in local school government (e.g. parent council or school management committee)',
 'School size (Sum)']

In [7]:
sch_df.columns

Index(['CNT', 'CNTSCHID', 'SC064Q05WA', 'SC064Q06WA', 'SC064Q01TA',
       'SC064Q02TA', 'SC064Q04NA', 'SC064Q03TA', 'SCHSIZE'],
      dtype='object')

In [ ]:
# ----------------------------------------------------------------------
# 4. Filter to the six target economies
# ----------------------------------------------------------------------
target_cnt = ["JPN", "KOR", "SGP", "TAP", "HKG", "MAC"]
stu_df = stu_df[stu_df["CNT"].isin(target_cnt)]
sch_df = sch_df[sch_df["CNT"].isin(target_cnt)]


print("After filtering countries:")
print(stu_df["CNT"].value_counts())

After filtering countries:
CNT
SGP    6606
KOR    6454
HKG    5907
TAP    5857
JPN    5760
MAC    4384
Name: count, dtype: int64


In [9]:
# ----------------------------------------------------------------------
# 5. Merge school -> student
# ----------------------------------------------------------------------
merge_keys = ["CNT", "CNTSCHID"]
merged = stu_df.merge(sch_df, on=merge_keys, how="left")
print("Merged dataset shape:", merged.shape)

Merged dataset shape: (34968, 26)


In [26]:
# Pairwise deletion – drop only rows with missing model vars
model_vars = [
    "ST268Q01JA","ST268Q04JA","ST268Q07JA",
    "SC064Q01TA","SC064Q02TA","SC064Q03TA","SC064Q04NA","SC064Q05WA","SC064Q06WA",
    "PV1MATH","PV2MATH","PV3MATH","PV4MATH","PV5MATH","PV6MATH","PV7MATH","PV8MATH","PV9MATH","PV10MATH"
]
merged = merged.dropna(subset=model_vars, how="any")

In [27]:
# ----------------------------------------------------------------------
# 6. Optional cleaning & recoding
# ----------------------------------------------------------------------

# Gender: 1=Female, 2=Male  → Female=1, Male=0
merged["female"] = np.where(merged["ST004D01T"] == 1, 1, 0)

# Standardize continuous controls
for v in ["ESCS", "ST001D01T", "SCHSIZE"]:
    merged[v + "_z"] = (merged[v] - merged[v].mean()) / merged[v].std()

# Drop rows missing key SEM indicators 
merged = merged.dropna(subset=[
    "ST268Q01JA","ST268Q04JA","ST268Q07JA",
    "PV1MATH","SC064Q01TA"
])

print("After dropping missing:", merged.shape)

After dropping missing: (28368, 30)


In [28]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri

In [29]:
# Convert to R dataframe
with pandas2ri.converter.context():
    r_df = pandas2ri.py2rpy(merged)


In [30]:
# Send to R environment
ro.globalenv["pisa_dat"] = r_df

# Load R packages
ro.r('library(lavaan)')
ro.r('library(psych)')

print("✅ DataFrame successfully sent to R environment as `pisa_dat`")

✅ DataFrame successfully sent to R environment as `pisa_dat`


In [31]:
lavaan_model = """
# -------------------------------
# Measurement model
# -------------------------------
SMP =~ PV1MATH + PV2MATH + PV3MATH + PV4MATH + PV5MATH +
       PV6MATH + PV7MATH + PV8MATH + PV9MATH + PV10MATH

SMS =~ ST268Q01JA + ST268Q04JA + ST268Q07JA

SPI =~ SC064Q01TA + SC064Q02TA + SC064Q03TA + SC064Q04NA +
       SC064Q05WA + SC064Q06WA

# -------------------------------
# Structural model
# -------------------------------
SMS ~ a*SPI + c1*female + c2*ESCS_z + c3*ST001D01T_z + c4*MISCED + c5*SCHSIZE_z
SMP ~ b*SMS + cprime*SPI + d1*female + d2*ESCS_z + d3*ST001D01T_z + d4*MISCED + d5*SCHSIZE_z

# Indirect and total effects
indirect := a*b
total := cprime + (a*b)
"""


In [32]:
ro.r.assign("model_string", lavaan_model)
ro.r('fit <- sem(model_string, data=pisa_dat, std.lv=TRUE, missing="fiml")')
summary = ro.r('summary(fit, standardized=TRUE, fit.measures=TRUE, rsquare=TRUE)')
print(summary)


R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1: lavaan->lav_data_full():  
   some observed variances are (at least) a factor 1000 times larger than 
   others; use varTable(fit) to investigate 
  
R callback write-console: 2: lavaan->lav_data_full():  
   327 cases were deleted due to missing values in exogenous variable(s), 
   while fixed.x = TRUE. 
  


lavaan 0.6-20 ended normally after 125 iterations

  Estimator                                         ML
  Optimization method                           NLMINB
  Number of model parameters                        70

                                                  Used       Total
  Number of observations                         28041       28368
  Number of missing patterns                         1            

Model Test User Model:
                                                       
  Test statistic                              24258.034
  Degrees of freedom                                234
  P-value (Chi-square)                            0.000

Model Test Baseline Model:

  Test statistic                            698324.682
  Degrees of freedom                               266
  P-value                                        0.000

User Model versus Baseline Model:

  Comparative Fit Index (CFI)                    0.966
  Tucker-Lewis Index (TLI)                       

In [33]:
ro.r('''
write.csv(parameterEstimates(fit, standardized = TRUE, boot.ci.type = "bca.simple"),
          file = "SEM_results.csv", row.names = FALSE)
''')
print("✅ Saved: SEM_results.csv")


✅ Saved: SEM_results.csv


In [34]:
# Fit SEM (example for real dataset)
ro.r.assign("model_string", lavaan_model)
ro.r.assign("pisa_dat", r_df)
ro.r('fit <- sem(model_string, data=pisa_dat, std.lv=TRUE, missing="fiml")')


R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1: lavaan->lav_data_full():  
   some observed variances are (at least) a factor 1000 times larger than 
   others; use varTable(fit) to investigate 
  
R callback write-console: 2: lavaan->lav_data_full():  
   327 cases were deleted due to missing values in exogenous variable(s), 
   while fixed.x = TRUE. 
  


# Structural Coherence baseline

In [35]:
from pingouin import cronbach_alpha

# === 1. Extract standardized loadings and correlations ===
# Get standardized solution table
std_sol = ro.r('parameterEstimates(fit, standardized=TRUE)')
std_df = pandas2ri.rpy2py(std_sol)

# Keep only loadings (latent =~ indicator)
loadings = std_df[std_df['op'] == '=~'][['lhs', 'rhs', 'std.all']]
loadings.columns = ['latent', 'indicator', 'loading']

# === 2. Compute Cronbach’s α, CR, AVE per construct ===
results = []

for latent in loadings['latent'].unique():
    subset = loadings[loadings['latent'] == latent]
    indicators = subset['indicator'].tolist()
    lambda_vals = subset['loading'].values.astype(float)

    # Composite Reliability (CR)
    CR = (np.sum(lambda_vals)**2) / ((np.sum(lambda_vals)**2) + np.sum(1 - lambda_vals**2))

    # Average Variance Extracted (AVE)
    AVE = np.sum(lambda_vals**2) / len(lambda_vals)

    # Cronbach’s α from raw data
    try:
        df_sub = merged[indicators].dropna()
        alpha, _ = cronbach_alpha(df_sub)
    except Exception:
        alpha = np.nan

    results.append({
        'Construct': latent,
        'Indicators': ', '.join(indicators),
        'Cronbach_alpha': round(alpha, 3),
        'Composite_Reliability': round(CR, 3),
        'AVE': round(AVE, 3)
    })

reliability_df = pd.DataFrame(results)
print("\n=== Reliability & Validity Metrics ===")
print(reliability_df)
reliability_df.to_csv("reliability_baseline.csv", index=False)
print("✅ reliability_baseline.csv saved")

# === 3. Compute discriminant validity (Fornell–Larcker) ===
# Extract latent correlation matrix from lavaan
lat_cor = ro.r('lavInspect(fit, "cor.lv")')

# Convert to numpy array
lat_cor_arr = np.array(pandas2ri.rpy2py(lat_cor))

# Get row and column names from R object
lat_names = list(ro.r('rownames(lavInspect(fit, "cor.lv"))'))
lat_cor_df = pd.DataFrame(lat_cor_arr, columns=lat_names, index=lat_names)

print("\n=== Latent Variable Correlation Matrix ===")
print(lat_cor_df.round(3))


fornell_df = pd.DataFrame({
    'Construct': reliability_df['Construct'],
    'SQRT_AVE': np.sqrt(reliability_df['AVE']).round(3)
})
print("\n=== Fornell–Larcker Criterion (√AVE vs correlations) ===")
print(fornell_df)
print(lat_cor_df.round(3))

lat_cor_df.to_csv("latent_correlation_matrix.csv", index=False)
print("✅ latent correlation matrix.csv saved")

fornell_df.to_csv("Fornell_Larcker.csv", index=False)
print("✅ Fornell_Larcker.csv saved")

# === 4. (Optional) Compare Real vs Synthetic congruence ===
# Example: Assuming you computed loadings for both fits (real, synthetic)
# merge loadings tables by latent+indicator
def tucker_congruence(load_real, load_synth):
    merged = pd.merge(load_real, load_synth, on=['latent', 'indicator'], suffixes=('_real', '_synth'))
    scores = []
    for latent in merged['latent'].unique():
        sub = merged[merged['latent'] == latent]
        num = np.sum(sub['loading_real'] * sub['loading_synth'])
        denom = np.sqrt(np.sum(sub['loading_real']**2) * np.sum(sub['loading_synth']**2))
        rho = num / denom if denom != 0 else np.nan
        scores.append({'Construct': latent, 'Tucker_Congruence': round(rho, 3)})
    return pd.DataFrame(scores)

# Example usage (when you have synthetic SEM result)
# congruence_df = tucker_congruence(loadings_real, loadings_synth)
# print(congruence_df)



=== Reliability & Validity Metrics ===
  Construct                                         Indicators  \
0       SMP  PV1MATH, PV2MATH, PV3MATH, PV4MATH, PV5MATH, P...   
1       SMS                 ST268Q01JA, ST268Q04JA, ST268Q07JA   
2       SPI  SC064Q01TA, SC064Q02TA, SC064Q03TA, SC064Q04NA...   

   Cronbach_alpha  Composite_Reliability    AVE  
0           0.992                  0.992  0.923  
1           0.718                  0.761  0.541  
2           0.781                  0.778  0.401  
✅ reliability_baseline.csv saved

=== Latent Variable Correlation Matrix ===
       SMP    SMS    SPI
SMP  1.000  0.394  0.086
SMS  0.394  1.000  0.016
SPI  0.086  0.016  1.000

=== Fornell–Larcker Criterion (√AVE vs correlations) ===
  Construct  SQRT_AVE
0       SMP     0.961
1       SMS     0.736
2       SPI     0.633
       SMP    SMS    SPI
SMP  1.000  0.394  0.086
SMS  0.394  1.000  0.016
SPI  0.086  0.016  1.000
✅ latent correlation matrix.csv saved
✅ Fornell_Larcker.csv saved


# Generate Statistical Fidelity baseline

In [36]:
from scipy.stats import skew, kurtosis

# 1. Select key numeric/ordinal variables
core_vars = [
    'PV1MATH','PV2MATH','PV3MATH','PV4MATH','PV5MATH',
    'PV6MATH','PV7MATH','PV8MATH','PV9MATH','PV10MATH',
    'ST268Q01JA','ST268Q04JA','ST268Q07JA',
    'SC064Q01TA','SC064Q02TA','SC064Q03TA','SC064Q04NA',
    'SC064Q05WA','SC064Q06WA',
    'MISCED','ESCS','ESCS_z','ST001D01T','SCHSIZE','SCHSIZE_z'
]

df_core = merged[core_vars].copy()

# 2. Descriptive stats
desc = df_core.describe().T
desc["skewness"]   = df_core.skew(numeric_only=True)
desc["kurtosis"]   = df_core.kurtosis(numeric_only=True)
desc = desc[["mean","std","min","25%","50%","75%","max","skewness","kurtosis"]]

# 3. Correlation matrix
corr_matrix = df_core.corr()

# 4. Save baseline summaries
desc.to_csv("PISA2022_statistical_baseline_summary.csv")
corr_matrix.to_csv("PISA2022_statistical_baseline_corr.csv")

print("✅ Baseline summaries saved:")
print(" - Descriptives: PISA2022_statistical_baseline_summary.csv")
print(" - Correlations: PISA2022_statistical_baseline_corr.csv")


✅ Baseline summaries saved:
 - Descriptives: PISA2022_statistical_baseline_summary.csv
 - Correlations: PISA2022_statistical_baseline_corr.csv


# Semantic Baseline

In [37]:
# --- Student metadata ---
stu_schema = pd.DataFrame({
    "Variable": stu_meta.column_names,
    "Label": stu_meta.column_labels
})
stu_schema["Source"] = "Student"

# --- School metadata ---
sch_schema = pd.DataFrame({
    "Variable": sch_meta.column_names,
    "Label": sch_meta.column_labels
})

sch_schema["Source"] = "School"

# Combine metadata
semantic_schema_auto = pd.concat([stu_schema, sch_schema], ignore_index=True)

# --- Infer data type heuristically ---
def infer_type(var):
    if var.startswith("PV"):
        return "Continuous (Plausible Value)"
    if var.startswith(("ST", "SC")):
        return "Ordinal / Likert"
    if "ESCS" in var:
        return "Continuous (Index)"
    if "MISCED" in var:
        return "Ordinal (Education level)"
    if "ST004D01T" in var:
        return "Binary (Gender: 1=Female, 2=Male)"
    if "SCHSIZE" in var:
        return "Continuous (School size)"
    return "Other / Mixed"

semantic_schema_auto["Inferred_Type"] = semantic_schema_auto["Variable"].apply(infer_type)

# Filter only variables used in merged dataset
used_vars = merged.columns.tolist()
semantic_schema_auto = semantic_schema_auto[semantic_schema_auto["Variable"].isin(used_vars)]

# Save semantic schema
semantic_schema_auto.to_csv("PISA2022_semantic_schema_auto.csv", index=False)
print("✅ Auto-generated semantic schema saved to: PISA2022_semantic_schema_auto.csv")

# Preview
semantic_schema_auto


✅ Auto-generated semantic schema saved to: PISA2022_semantic_schema_auto.csv


,Variable,Label,Source,Inferred_Type
0,CNT,Country code 3-character,Student,Other / Mixed
1,CNTSCHID,Intl. School ID,Student,Other / Mixed
2,ST001D01T,Student International Grade (Derived),Student,Ordinal / Likert
3,ST004D01T,Student (Standardized) Gender,Student,Ordinal / Likert
4,ST268Q01JA,Agree/disagree: Mathematics is one of my favou...,Student,Ordinal / Likert
5,ST268Q04JA,Agree/disagree: Mathematics is easy for me.,Student,Ordinal / Likert
6,ST268Q07JA,Agree/disagree: I want to do well in my mathem...,Student,Ordinal / Likert
7,MISCED,Mother’s level of education (ISCED),Student,Ordinal (Education level)
8,ESCS,"Index of economic, social and cultural status",Student,Continuous (Index)
9,PV1MATH,Plausible Value 1 in Mathematics,Student,Continuous (Plausible Value)
